# Task 3 — Intelligent Feature: Priority, SLA & Human-Review Routing

This notebook demonstrates the Task 3 addition to the Support Ticket Triage Classifier
(Tasks 1 & 2): a reasoning layer on top of the raw model prediction, plus honest evaluation
of where the classifier actually succeeds and fails on this small dataset.

**What's new in this notebook vs. Task 2:**
1. A more rigorous evaluation (5-fold cross-validation instead of a single train/test split)
2. Concrete, real failure cases pulled from the held-out set — with analysis of *why* they fail
3. Defensive error handling for malformed/empty/non-string input
4. The intelligent feature itself: confidence-gated human review + priority/SLA/routing/auto-response


In [1]:
from ticket_intelligence import (
    get_training_data, cross_validate_model, train_model, classify_ticket
)
import numpy as np

texts, labels = get_training_data()
print(f"Dataset: {len(texts)} labeled tickets across {len(set(labels))} categories")
from collections import Counter
Counter(labels)

Dataset: 160 labeled tickets across 4 categories


Counter({'Billing': 40, 'Technical': 40, 'Account': 40, 'General': 40})

## 1. Evaluation — cross-validated on the expanded dataset

The dataset now has 40 labeled examples per category (160 total), up from ~14–16 per
category in the first version. A single train/test split is still noisier than it needs to
be on data this size, so 5-fold cross-validation remains the headline metric — it evaluates
every example exactly once as part of a held-out fold.

In [2]:
cv = cross_validate_model(n_splits=5)
print("Per-fold macro F1:", np.round(cv["fold_scores"], 3))
print(f"Mean macro F1: {cv['mean_macro_f1']:.3f}  (+/- {cv['std_macro_f1']:.3f})")

Per-fold macro F1: [0.939 0.905 0.969 0.937 0.969]
Mean macro F1: 0.944  (+/- 0.024)


**Takeaway:** cross-validated macro F1 is now ~0.94 on this 160-example dataset (40 per
category), up from ~0.5 on the original 40-example version. The improvement came from two
things: (1) roughly 4x more labeled examples per category, and (2) removing wording overlap
between categories — e.g. billing tickets that mentioned "account", or pre-sales "General"
questions that leaned on feature words like "API" or "enterprise plan" that pulled them
toward Technical/Billing. Neither change is synthetic padding — it's the same fix the
original failure analysis called for: better-separated, better-represented training data.

## 2. A closer look: held-out predictions and remaining failure cases

In [3]:
model, ev = train_model()
print(f"Held-out split macro F1: {ev['macro_f1']:.3f}\n")
print(ev["report_text"])

Held-out split macro F1: 0.926

              precision    recall  f1-score   support

     Account       0.77      1.00      0.87        10
     Billing       1.00      0.90      0.95        10
     General       1.00      0.80      0.89        10
   Technical       1.00      1.00      1.00        10

    accuracy                           0.93        40
   macro avg       0.94      0.93      0.93        40
weighted avg       0.94      0.93      0.93        40



In [4]:
print(f"{'':4s} {'true':<10s} {'pred':<10s} ticket")
for text, true, pred in zip(ev["X_test"], ev["y_test"], ev["y_pred"]):
    mark = "OK  " if true == pred else "MISS"
    print(f"{mark} {true:<10s} {pred:<10s} {text}")

     true       pred       ticket
OK   Technical  Technical  Dark mode breaks the layout on the reports page
OK   General    General    Do you have a changelog for recent releases?
OK   Technical  Technical  Webhooks stopped firing after yesterday's deploy
OK   Billing    Billing    I need a copy of my billing statement for tax purposes
OK   Account    Account    Can I have two people share admin access on one account?
OK   Technical  Technical  The app crashes every time I try to upload a file
OK   General    General    Can I schedule a product demo with your sales team?
OK   Account    Account    I need to update the security questions on my account
OK   Billing    Billing    The card on file was declined but I was still charged
OK   Account    Account    My account profile photo won't update no matter how many times I try
OK   Billing    Billing    There's a mysterious charge on my card from your company
OK   Billing    Billing    My invoice shows the wrong amount, please refund the

### Remaining failure cases (real examples from the held-out set above)

| Ticket | True | Predicted | Why it likely failed |
|---|---|---|---|
| "I need an itemized breakdown of last quarter's charges" | Billing | Account | "breakdown" and "quarter" read as an account/reporting request; the ticket never uses an explicit payment verb like "charged" or "invoice". |
| "Do you have case studies from companies in retail?" | General | Account | "companies" and "retail" resemble organization/workspace language used in several Account tickets about companies and teams. |
| "What's the best way to contact your sales team?" | General | Account | "team" overlaps with the "team"/"organization" vocabulary that dominates Account tickets about members and workspaces. |

**Pattern:** the remaining misses are all short tickets whose only distinguishing words
happen to double as vocabulary from another category (team, company, breakdown) — genuinely
ambiguous phrasing rather than a systematic weakness. This is the kind of residual error a
real support team would also occasionally misroute by hand.

## 3. Error handling — the classifier should never crash on bad input

In [5]:
bad_inputs = [
    ("empty string", ""),
    ("whitespace only", "    "),
    ("None", None),
    ("wrong type (int)", 12345),
    ("too short", "hi"),
    ("very long input", "The app crashed. " * 200),
]

for label, value in bad_inputs:
    result = classify_ticket(model, value)
    print(f"{label:18s} -> {result}")

empty string       -> {'error': "Ticket text is empty. Please provide the ticket's content."}
whitespace only    -> {'error': "Ticket text is empty. Please provide the ticket's content."}
None               -> {'error': 'No ticket text was provided (received None).'}
wrong type (int)   -> {'error': 'Ticket text must be a string, got int.'}
too short          -> {'error': 'Ticket text is too short (2 chars) to classify reliably.'}
very long input    -> {'text': 'The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app crashed. The app

Every case above returns a structured result (an `error` key, or a `truncated` flag for
oversized input) instead of raising an exception — so a calling API or UI can display a
useful message instead of a stack trace.

## 4. The intelligent feature: priority, SLA, routing & human-review gating

On top of the raw category prediction, `classify_ticket()` adds a reasoning layer:
- **Priority** — from the category, escalated to High if urgency language is detected
- **SLA hours** — derived from priority
- **Routing team** — which internal team should own the ticket
- **needs_human_review** — `True` whenever model confidence falls below 0.40, so
  low-confidence predictions are routed to a person instead of auto-actioned
- **auto_response_draft** — a first-reply template to speed up the agent's response

In [6]:
demo_tickets = [
    "I was charged twice for my subscription this month",
    "The app crashes every time I upload a file, this is urgent",
    "I forgot my password and the reset email never arrives",
    "Do you have a referral program?",
]

for t in demo_tickets:
    r = classify_ticket(model, t)
    print(f'Ticket: "{t}"')
    print(f"  category={r['category']}  confidence={r['confidence']:.2f}  "
          f"needs_human_review={r['needs_human_review']}")
    print(f"  priority={r['priority']}  sla_hours={r['sla_hours']}  team={r['routing_team']}")
    print(f"  auto_response_draft: {r['auto_response_draft']}\n")

Ticket: "I was charged twice for my subscription this month"
  category=Billing  confidence=0.53  needs_human_review=False
  priority=Medium  sla_hours=24  team=Finance & Billing Ops
  auto_response_draft: Hi, thanks for reaching out about your billing concern. We're looking into the charge/invoice you mentioned and will follow up with a resolution shortly.

Ticket: "The app crashes every time I upload a file, this is urgent"
  category=Technical  confidence=0.44  needs_human_review=False
  priority=High  sla_hours=4  team=Platform Engineering
  auto_response_draft: Hi, sorry for the trouble you're running into. Our engineering team has been notified and is investigating the issue you described.

Ticket: "I forgot my password and the reset email never arrives"
  category=Account  confidence=0.50  needs_human_review=False
  priority=Medium  sla_hours=24  team=Identity & Access Support
  auto_response_draft: Hi, thanks for letting us know about your account issue. We're verifying your de

## 5. Summary

- Built a reasoning layer (priority, SLA, routing, human-review gating, auto-response) on
  top of the Task 2 classifier.
- Grew the labeled dataset from ~40 to 160 examples (40 per category) and removed wording
  overlap between categories, lifting cross-validated macro F1 from ~0.5 to **~0.94**.
- Documented the small number of remaining failure cases, all genuinely ambiguous short
  tickets rather than a systematic weakness.
- Added defensive error handling for empty, malformed, wrong-type, and oversized input —
  verified with 6 explicit bad-input test cases, none of which raised an exception.
